# AQI Category Prediction – Mini Project
Predict the **AQI category** (Good … Severe) of an Indian city-day from air-pollutant readings.

Steps: 1) Load & explore → 2) Clean → 3) Features → 4) Split → 5) Train 4 models → 6) Compare → 7) Save

In [ ]:
import os, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
os.makedirs("figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

from preprocess import *      # our own cleaning / feature code (preprocess.py)

## 1. Load and explore the data

In [ ]:
raw = pd.read_csv("city_day.csv")
print("Shape:", raw.shape)
raw.head()

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
missing = (raw.isna().mean() * 100).sort_values(ascending=False).round(1)
print(missing)

plt.figure(figsize=(7, 5))
sns.barplot(x=missing.values, y=missing.index, color="tomato")
plt.xlabel("% missing"); plt.title("Missing values per column")
plt.tight_layout(); plt.savefig("figures/missing_values.png", dpi=150); plt.show()

In [ ]:
order = CLASS_ORDER
plt.figure(figsize=(7, 4))
sns.countplot(x="AQI_Bucket", data=raw, order=order, color="steelblue")
plt.title("AQI category distribution"); plt.xticks(rotation=25)
plt.tight_layout(); plt.savefig("figures/class_distribution.png", dpi=150); plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(raw[POLLUTANTS + ["AQI"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap"); plt.tight_layout()
plt.savefig("figures/correlation.png", dpi=150); plt.show()

## 2. Data cleaning
- Remove duplicate rows
- Remove rows where the target (`AQI_Bucket`) is missing
- Sort by city and date
- Fill missing pollutant values from neighbouring days of the **same city** (interpolation)

In [ ]:
df = clean_data(raw)
print("Rows before:", len(raw), "-> after cleaning:", len(df))
df = fill_missing(df)
print("Missing values left in pollutants (filled later with median):")
print(df[POLLUTANTS].isna().sum().sum())

## 3. Feature creation
Pollution changes slowly, so for PM2.5, PM10, NO2 and CO we add **yesterday's value** and the **3-day average** (only past days are used). We also add the city code and month.

In [ ]:
df = add_features(df)
print(len(FEATURES), "features:", FEATURES)
X = df[FEATURES]
y = df["AQI_Bucket"]

## 4. Train / test split (80% / 20%, stratified)
Then the remaining missing values are filled with the **training** median, and the data is scaled (needed for Logistic Regression and KNN).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

train_median = X_train.median()
X_train = X_train.fillna(train_median)
X_test = X_test.fillna(train_median)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

## 5. Train 4 models
Logistic Regression and KNN (these use scaled data), Random Forest and Gradient Boosting (tree models, no scaling needed).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# name: (model, uses_scaled_data)
models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000), True),
    "KNN":                 (KNeighborsClassifier(n_neighbors=5), True),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), False),
    "Gradient Boosting":   (HistGradientBoostingClassifier(max_iter=600, learning_rate=0.03,
                                                           l2_regularization=1.0, random_state=42), False),
}

predictions, rows = {}, []
for name, (model, scaled) in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if scaled else (X_train, X_test)
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    predictions[name] = pred
    rows.append({"Model": name,
                 "Accuracy":  accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred, average="weighted"),
                 "Recall":    recall_score(y_test, pred, average="weighted"),
                 "F1 Score":  f1_score(y_test, pred, average="weighted"),
                 "Macro F1":  f1_score(y_test, pred, average="macro")})
    print("done:", name)

## 6. Compare the models

In [ ]:
results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
results.round(4).to_csv("results.csv", index=False)
results.round(4)

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=results, x="Model", y="Accuracy", color="steelblue")
for p in ax.patches:
    ax.annotate(f"{p.get_height()*100:.1f}%", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.axhline(0.85, color="red", linestyle="--", label="85% target")
plt.ylim(0, 1); plt.xticks(rotation=20); plt.legend()
plt.title("Model accuracy comparison")
plt.tight_layout(); plt.savefig("figures/model_accuracy.png", dpi=150); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

best = results.loc[0, "Model"]
print("Best model:", best)
print(classification_report(y_test, predictions[best], labels=CLASS_ORDER))

# confusion matrix of every model
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, name in zip(axes.ravel(), models):
    cm = confusion_matrix(y_test, predictions[name], labels=CLASS_ORDER)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=40)
    ax.set_title(name)
plt.tight_layout(); plt.savefig("figures/confusion_matrices_all.png", dpi=150); plt.show()

# big confusion matrix of the best model
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, predictions[best], labels=CLASS_ORDER)
ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
ax.set_title("Confusion matrix - " + best)
plt.tight_layout(); plt.savefig("figures/confusion_matrix_best.png", dpi=150); plt.show()

In [ ]:
# Which features matter most? (Random Forest)
rf = models["Random Forest"][0]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values().tail(12)
imp.plot(kind="barh", figsize=(8, 5), color="seagreen", title="Top 12 features (Random Forest)")
plt.tight_layout(); plt.savefig("figures/feature_importance.png", dpi=150); plt.show()

## 7. Save everything for the Streamlit app

In [ ]:
for name, (model, scaled) in models.items():
    joblib.dump(model, "models/" + name.replace(" ", "_") + ".pkl", compress=3)  # compress keeps files small for GitHub
joblib.dump({"scaler": scaler, "median": train_median, "scaled_models": [n for n, (m, s) in models.items() if s]},
            "models/preprocessing.pkl")
print("Saved. Best model:", best, "- accuracy: %.2f%%" % (results.loc[0, "Accuracy"] * 100))